# Лабораторная работа №4

В рамках данной лабораторной работы необходимо разработать масштабируемую программную систему на языке Python с использованием принципов объектно-ориентированного программирования. Проект должен продемонстрировать умение проектировать архитектуру приложения, применять абстракцию, наследование, инкапсуляцию и полиморфизм, а также реализовывать расширяемую и логически структурированную модель предметной области.

### Архитектурное проектирование

Перед началом реализации необходимо определить предметную область и выделить ключевые сущности системы. Следует продумать структуру классов, их роли и взаимосвязи, избегая дублирования логики.

Обязательно предоставить UML-диаграмму классов с указанием атрибутов, методов и типов связей. Также требуется кратко обосновать выбранную архитектуру и объяснить, каким образом она обеспечивает расширяемость системы.


### Абстрактные классы (ABC)

В системе должно быть минимум 2 абстрактных класса с использованием `abc.ABC` и минимум 2 абстрактных методов в каждом. Абстрактные классы должны задавать общий интерфейс и описывать поведение, которое обязательно для всех наследников.

Создание экземпляров абстрактных классов запрещено. Конкретные классы обязаны полностью реализовать все абстрактные методы.

### Наследование

Необходимо реализовать минимум 3 уровня наследования и не менее 5 конкретных классов-наследников. Иерархия должна быть логически обоснованной и отражать реальную структуру предметной области.

Дочерние классы должны переопределять методы базового класса и при необходимости расширять их функциональность с использованием `super()`. Наследование не должно использоваться формально — только там, где существует отношение «является».


### Инкапсуляция

В системе должны использоваться защищённые или приватные атрибуты для ограничения прямого доступа к данным. Управление состоянием объектов должно происходить через методы и свойства.

Обязательно реализовать минимум 3 свойства с использованием `@property` и сеттеры с валидацией значений. Некорректные данные должны вызывать исключения.


### Полиморфизм

Необходимо реализовать методы с одинаковым именем, но различной реализацией в дочерних классах. Это должно демонстрировать единый интерфейс для разных типов объектов.

Работа с такими объектами должна осуществляться через ссылку на базовый тип, например в списке объектов. Поведение программы должно изменяться в зависимости от конкретного типа объекта.

### Магические методы

Реализовать минимум 4 магических метода (`__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__add__`, и др.). Их применение должно быть логически оправдано и интегрировано в работу системы.

Методы должны использоваться для удобства взаимодействия с объектами: вывода, сравнения, сортировки или объединения. Формальная реализация без практического применения не засчитывается.

### Пользовательские исключения

Создать одно базовое исключение системы и минимум 3 специализированных исключения. Они должны описывать ошибки бизнес-логики или нарушения ограничений системы.

Исключения необходимо выбрасывать через `raise` и обрабатывать через `try/except`. В программе должна быть продемонстрирована корректная реакция системы на ошибки.


### Работа с файлами

Реализовать сохранение и загрузку состояния системы в формате JSON или CSV. Должна быть предусмотрена сериализация объектов и их корректное восстановление.

После загрузки система должна продолжать работу без потери данных и логических связей между объектами. Необходимо обработать возможные ошибки чтения или записи файлов.


###  Минимальные требования

* ≥ 12 классов
* ≥ 2 абстрактных класса
* ≥ 3 уровня наследования
* ≥ 4 магических метода
* ≥ 3 пользовательских исключения



In [6]:
import abc
import json


# ==========================================
# ПОЛЬЗОВАТЕЛЬСКИЕ ИСКЛЮЧЕНИЯ
# ==========================================
class FleetException(Exception):
    """Базовое исключение для всей системы автопарка"""

    pass


class InvalidPayloadException(FleetException):
    """Вызывается при превышении лимита грузоподъемности"""

    pass


class InsufficientRangeException(FleetException):
    """Вызывается, если запаса хода транспорта не хватает на маршрут"""

    pass


class NegativeSalaryException(FleetException):
    """Вызывается при попытке установить отрицательную зарплату"""

    pass


# ==========================================
# АБСТРАКТНЫЕ КЛАССЫ (ABC)
# ==========================================
class Vehicle(abc.ABC):
    """Абстрактный класс первого уровня для всего транспорта"""

    def __init__(self, vehicle_id: str, capacity: float):
        self._vehicle_id = vehicle_id
        self._capacity = capacity  # Грузоподъемность в кг

    @property
    def vehicle_id(self):
        return self._vehicle_id

    @property
    def capacity(self):
        return self._capacity

    @abc.abstractmethod
    def calculate_max_range(self) -> float:
        """Возвращает максимальный запас хода в км"""
        pass

    @abc.abstractmethod
    def get_vehicle_type(self) -> str:
        """Возвращает строковое описание типа транспорта"""
        pass


class MotorizedVehicle(Vehicle, abc.ABC):
    """Абстрактный класс второго уровня для моторного транспорта"""

    def __init__(self, vehicle_id: str, capacity: float, fuel_efficiency: float):
        super().__init__(vehicle_id, capacity)
        self._fuel_efficiency = fuel_efficiency  # Расход на 100 км или мин/км

    @abc.abstractmethod
    def refuel(self, amount: float):
        """Заправить или зарядить транспортное средство"""
        pass


class Employee(abc.ABC):
    """Абстрактный класс для сотрудников компании"""

    def __init__(self, name: str, emp_id: str, salary: float):
        self._name = name
        self._emp_id = emp_id
        if salary < 0:
            raise NegativeSalaryException("Зарплата не может быть отрицательной")
        self._salary = salary

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, value: float):
        if value < 0:
            raise NegativeSalaryException("Зарплата не может быть отрицательной")
        self._salary = value

    @abc.abstractmethod
    def get_role(self) -> str:
        pass

    @abc.abstractmethod
    def perform_duty(self) -> str:
        pass


# ==========================================
# КОНКРЕТНЫЕ КЛАССЫ-НАСЛЕДНИКИ (ТРАНСПОРТ)
# ==========================================
class DeliveryVan(MotorizedVehicle):
    """Конкретный класс третьего уровня - Грузовой фургон"""

    def __init__(
        self,
        vehicle_id: str,
        capacity: float,
        fuel_efficiency: float,
        tank_capacity: float,
    ):
        super().__init__(vehicle_id, capacity, fuel_efficiency)
        self._tank_capacity = tank_capacity
        self._current_fuel = tank_capacity

    def calculate_max_range(self) -> float:
        return (self._current_fuel / self._fuel_efficiency) * 100

    def get_vehicle_type(self) -> str:
        return "Грузовой фургон"

    def refuel(self, amount: float):
        self._current_fuel = min(self._tank_capacity, self._current_fuel + amount)


class ElectricBike(MotorizedVehicle):
    """Конкретный класс третьего уровня - Электровелосипед"""

    def __init__(
        self,
        vehicle_id: str,
        capacity: float,
        fuel_efficiency: float,
        battery_kwh: float,
    ):
        super().__init__(vehicle_id, capacity, fuel_efficiency)
        self._battery_kwh = battery_kwh
        self._current_charge = battery_kwh

    def calculate_max_range(self) -> float:
        return self._current_charge * self._fuel_efficiency

    def get_vehicle_type(self) -> str:
        return "Электровелосипед"

    def refuel(self, amount: float):
        self._current_charge = min(self._battery_kwh, self._current_charge + amount)


class HeavyTruck(MotorizedVehicle):
    """Конкретный класс третьего уровня - Тяжелый грузовик"""

    def __init__(
        self,
        vehicle_id: str,
        capacity: float,
        fuel_efficiency: float,
        tank_capacity: float,
    ):
        super().__init__(vehicle_id, capacity, fuel_efficiency)
        self._tank_capacity = tank_capacity
        self._current_fuel = tank_capacity

    def calculate_max_range(self) -> float:
        return (self._current_fuel / self._fuel_efficiency) * 100

    def get_vehicle_type(self) -> str:
        return "Тяжелый грузовик"

    def refuel(self, amount: float):
        self._current_fuel = min(self._tank_capacity, self._current_fuel + amount)


# ==========================================
# КОНКРЕТНЫЕ КЛАССЫ-НАСЛЕДНИКИ (СОТРУДНИКИ)
# ==========================================
class Courier(Employee):
    """Курьер, осуществляющий доставку"""

    def get_role(self) -> str:
        return "Курьер"

    def perform_duty(self) -> str:
        return f"Курьер {self._name} доставляет заказы"


class Manager(Employee):
    """Менеджер, управляющий логистикой"""

    def get_role(self) -> str:
        return "Менеджер"

    def perform_duty(self) -> str:
        return f"Менеджер {self._name} оптимизирует маршруты"


# ==========================================
# ВСПОМОГАТЕЛЬНЫЕ И БИЗНЕС-КЛАССЫ
# ==========================================
class Order:
    """Класс заказа"""

    def __init__(self, order_id: str, weight: float, destination: str):
        self.order_id = order_id
        self.weight = weight
        self.destination = destination


class Route:
    """Класс маршрута доставки"""

    def __init__(self, route_id: str, distance: float):
        self.route_id = route_id
        self._distance = distance

    @property
    def distance(self):
        return self._distance


class Fleet:
    """Класс управления автопарком (основной контейнер)"""

    def __init__(self):
        self._vehicles = []

    def add_vehicle(self, vehicle: Vehicle):
        self._vehicles.append(vehicle)

    # МАГИЧЕСКИЕ МЕТОДЫ
    def __len__(self):
        return len(self._vehicles)

    def __getitem__(self, position):
        return self._vehicles[position]

    def __str__(self):
        return f"Автопарк: {len(self._vehicles)} транспортных средств"

    def __repr__(self):
        return f"Fleet(vehicles_count={len(self._vehicles)})"

    # ПОЛИМОРФИЗМ В ДЕЙСТВИИ
    def check_readiness(self, route: Route, cargo_weight: float):
        """Проверяет весь транспорт на пригодность к рейсу"""
        print(f"--- Проверка транспорта для маршрута {route.distance} км ---")
        for v in self._vehicles:
            try:
                if cargo_weight > v.capacity:
                    raise InvalidPayloadException(
                        f"Технический лимит превышен для {v.vehicle_id}"
                    )
                if route.distance > v.calculate_max_range():
                    raise InsufficientRangeException(
                        f"Недостаточно запаса хода для {v.vehicle_id}"
                    )
                print(
                    f"{v.vehicle_id} ({v.get_vehicle_type()}): Готов к рейсу!"
                )
            except FleetException as e:
                print(f"{v.vehicle_id} отклонен: {e}")

    # РАБОТА С ФАЙЛАМИ (JSON СЕРИАЛИЗАЦИЯ)
    def save_to_file(self, filename: str):
        data = []
        for v in self._vehicles:
            data.append(
                {
                    "id": v.vehicle_id,
                    "type": v.get_vehicle_type(),
                    "capacity": v.capacity,
                }
            )
        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4)
            print(f"Данные успешно сохранены в {filename}")
        except IOError as e:
            print(f"Ошибка записи файла: {e}")

    def load_from_file(self, filename: str):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                data = json.load(f)
            self._vehicles = []
            for item in data:
                # Восстановление объектов на основе сохраненного типа
                if item["type"] == "Грузовой фургон":
                    self.add_vehicle(
                        DeliveryVan(item["id"], item["capacity"], 12.0, 60.0)
                    )
                elif item["type"] == "Электровелосипед":
                    self.add_vehicle(
                        ElectricBike(item["id"], item["capacity"], 5.0, 2.0)
                    )
                elif item["type"] == "Тяжелый грузовик":
                    self.add_vehicle(
                        HeavyTruck(item["id"], item["capacity"], 25.0, 200.0)
                    )
            print(f"Данные успешно загружены из {filename}")
        except (IOError, json.JSONDecodeError) as e:
            print(f"Ошибка чтения файла: {e}")


# ==========================================
# ДЕМОНСТРАЦИЯ РАБОТЫ ПРОГРАММЫ
# ==========================================
if __name__ == "__main__":
    # Создание объектов автопарка
    my_fleet = Fleet()

    van = DeliveryVan("VAN-01", 1500.0, 10.0, 50.0)  # Запас хода 500 км
    bike = ElectricBike("BIKE-01", 80.0, 25.0, 2.0)  # Запас хода 50 км
    truck = HeavyTruck("TRUCK-01", 10000.0, 30.0, 300.0)  # Запас хода 1000 км

    my_fleet.add_vehicle(van)
    my_fleet.add_vehicle(bike)
    my_fleet.add_vehicle(truck)

    # Проверка магических методов
    print(str(my_fleet))
    print(repr(my_fleet))
    print(f"Первый транспорт в списке: {my_fleet[0].vehicle_id}")

    # Демонстрация валидации свойств и исключений
    try:
        bad_courier = Courier("Ivan", "C-01", -500.0)
    except NegativeSalaryException as error:
        print(f"Перехвачено исключение: {error}")

    # Бизнес-логика: проверка готовности к маршруту
    urgent_route = Route("R-01", 120.0)
    my_fleet.check_readiness(urgent_route, 100.0)

    # Сериализация / Десериализация
    filename = "fleet_state.json"
    my_fleet.save_to_file(filename)

    # Очищаем автопарк и восстанавливаем из файла
    new_fleet = Fleet()
    new_fleet.load_from_file(filename)
    print(f"Восстановлено транспортных средств: {len(new_fleet)}")


Автопарк: 3 транспортных средств
Fleet(vehicles_count=3)
Первый транспорт в списке: VAN-01
Перехвачено исключение: Зарплата не может быть отрицательной
--- Проверка транспорта для маршрута 120.0 км ---
VAN-01 (Грузовой фургон): Готов к рейсу!
BIKE-01 отклонен: Технический лимит превышен для BIKE-01
TRUCK-01 (Тяжелый грузовик): Готов к рейсу!
Данные успешно сохранены в fleet_state.json
Данные успешно загружены из fleet_state.json
Восстановлено транспортных средств: 3
